## Task 12 - Emotion Classification Pipeline (Video → Transcript → Translate → Classify → CSV)

### 1) Transcript using Assembly

In [1]:
video_dir = input("Please add the file location (audio/video) to transcribe: ").strip().strip('"').strip("'")

In [ ]:
"""
speech_to_text_assemblyAI.py

This script takes an MP3 audio file, transcribes it using AssemblyAI,
splits the text into sentences, and saves the sentences in an Excel file.
"""

import assemblyai as aai
import pandas as pd
import re
import os

def main(input_file, output_file="transcribed_data_assemblyAI.xlsx"):
    """
    Transcribe an mp3 audio file using AssemblyAI and save as xlsx.
    """
    # Set your API key
    aai.settings.api_key = "6f629fba6d0d42b48c9829a281c31bb6"

    
    # Optional: basic existence check
    if not os.path.isfile(input_file):
        raise FileNotFoundError(f"File not found: {input_file}")

    #  Create the transcriber object
    transcriber = aai.Transcriber()

    #  Transcribe the audio file (no language_code needed)
    transcript = transcriber.transcribe(input_file)

    # Extract text
    text = transcript.text

    # Split text into sentences
    text = re.sub(r'[!?]', '.', text)
    sentences = [s.strip() for s in text.split('.') if s.strip()]

    # Save to xlsx
    df = pd.DataFrame(sentences, columns=["Sentence"])
    df.to_excel(output_file, index=False)

    print(f"Transcription saved to {output_file}")

main(video_dir)

### 2) Translate EN → NL → EN

In [ ]:
"""
translate_en_nl_en.py

This script:
1. Loads sentences from the AssemblyAI transcript file (transcribed_data_assemblyAI.xlsx)
2. Translates each sentence EN -> NL -> EN
3. Saves the results into a new Excel file.
"""

import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from tqdm import tqdm

# Paths to saved models
EN_NL_MODEL_PATH = r"en_nl"
NL_EN_MODEL_PATH = r"nl_en"

# Load models and tokenizers
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading EN→NL model...")
en_nl_tokenizer = AutoTokenizer.from_pretrained(EN_NL_MODEL_PATH)
en_nl_model = AutoModelForSeq2SeqLM.from_pretrained(EN_NL_MODEL_PATH).to(device)

print("Loading NL→EN model...")
nl_en_tokenizer = AutoTokenizer.from_pretrained(NL_EN_MODEL_PATH)
nl_en_model = AutoModelForSeq2SeqLM.from_pretrained(NL_EN_MODEL_PATH).to(device)


def translate(text, tokenizer, model, max_new_tokens=200):
    """Simple helper for text translation"""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# Load transcribed sentences
input_file = r"transcribed_data_assemblyAI.xlsx"
df = pd.read_excel(input_file)

if "Sentence" not in df.columns:
    raise ValueError("Input file must have a 'Sentence' column")

# Perform translations
translations_nl = []
translations_back_en = []

for sentence in tqdm(df["Sentence"], desc="Translating EN→NL→EN"):
    # EN → NL
    nl_text = translate(sentence, en_nl_tokenizer, en_nl_model)
    # NL → EN
    en_text = translate(nl_text, nl_en_tokenizer, nl_en_model)
    translations_nl.append(nl_text)
    translations_back_en.append(en_text)


# Save results
df["Translation_NL"] = translations_nl
df["Translation_EN"] = translations_back_en

output_file = "translated_data.xlsx"
df.to_excel(output_file, index=False)
print(f"Translation saved to {output_file}")

### 3) Emotion Classification

In [ ]:
"""
emotion_classification_roberta.py

This script:
1. Loads the fine-tuned RoBERTa-Large model.
2. Reads the translated sentences file.
3. Predicts the emotion for each line.
4. Saves the final output in the required format:
    Sentence | Translation_NL | Translation_EN | Emotion
"""

# Inference on Translation_EN only; silent loading; save + show distribution
import os
import warnings
import torch
import pandas as pd
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification
from transformers.utils import logging as hf_logging

# Silence logs/warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()

# Paths
MODEL_PATH  = r"RoBERTa_Large_weighted"
BASE_MODEL  = "roberta-large"
INPUT_FILE  = r"translated_data.xlsx"
OUTPUT_FILE = "final_emotion_output.csv"

# Labels
emotion_labels = ['anger','disgust','fear','happiness','neutral','sadness','surprise']
label2id = {l:i for i,l in enumerate(emotion_labels)}
id2label = {i:l for i,l in enumerate(emotion_labels)}

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Config & tokenizer (quiet)
config = AutoConfig.from_pretrained(BASE_MODEL)
config.num_labels = len(emotion_labels)
config.label2id = label2id
config.id2label = id2label

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Build model from config (no base weights first)
model = AutoModelForSequenceClassification.from_config(config)

# Load fine-tuned weights and rename head keys
# Pick weight file
weight_file = None
for name in ("model.safetensors", "pytorch_model.bin"):
    p = os.path.join(MODEL_PATH, name)
    if os.path.exists(p):
        weight_file = p; break
if weight_file is None:
    raise FileNotFoundError("No model.safetensors or pytorch_model.bin found in MODEL_PATH.")

# Load raw state dict
if weight_file.endswith(".safetensors"):
    from safetensors.torch import load_file
    state = load_file(weight_file)
else:
    state = torch.load(weight_file, map_location="cpu")

# Rename Sequential-style classifier keys -> HF expected keys; drop unused pooler
rename = {
    "classifier.0.weight": "classifier.dense.weight",
    "classifier.0.bias":   "classifier.dense.bias",
    "classifier.3.weight": "classifier.out_proj.weight",
    "classifier.3.bias":   "classifier.out_proj.bias",
}
new_state = {}
for k, v in state.items():
    if k in ("roberta.pooler.dense.weight", "roberta.pooler.dense.bias"):
        continue
    new_state[rename.get(k, k)] = v

# Load weights
model.load_state_dict(new_state, strict=False)
model.to(device).eval()

# Load data
df = pd.read_excel(INPUT_FILE)

# Ensure columns exist (we will NOT use Sentence for prediction)
if "Sentence" not in df.columns:
    df["Sentence"] = ""
if "Translation_NL" not in df.columns:
    df["Translation_NL"] = ""
if "Translation_EN" not in df.columns:
    raise ValueError("Input must contain 'Translation_EN'.")

# Predict using Translation_EN only
texts = df["Translation_EN"].astype(str)
empty_mask = texts.str.strip().isin(["", "nan", "none"])

preds = ["neutral"] * len(texts)  # default for empties
idxs = (~empty_mask).to_numpy().nonzero()[0]

batch_size, max_length = 32, 128
for i in range(0, len(idxs), batch_size):
    idx = idxs[i:i+batch_size]
    chunk = texts.iloc[idx].tolist()
    enc = tokenizer(chunk, return_tensors="pt", truncation=True, padding=True, max_length=max_length)
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits
        ids = torch.argmax(logits, dim=1).tolist()
    for j, lab_id in zip(idx, ids):
        preds[j] = config.id2label[lab_id]

df["Emotion"] = preds

# Save required columns
out = df[["Sentence", "Translation_NL", "Translation_EN", "Emotion"]]
out.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

# Show prediction distribution (percentage)
dist = out["Emotion"].value_counts(normalize=True).mul(100).round(1)
print("Prediction distribution (%):")
print(dist.to_string())
print(f"\nSaved: {OUTPUT_FILE}")